In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [33]:
from numpy.linalg import inv
def implied_returns(covmat, weigths, delta=2.5):
    '''
    Computes the implied expected returns \Pi by reverse engineering the weights according to 
    the Black-Litterman model:
       \Pi = \delta \Sigma weigths
    Here, the inputs are:
    - delta, the risk aversion coefficient
    - covmat: variance-covariance matrix (N x N) as pd.DataFrame (\Sigma)
    - weigths: portfolio weights (N x 1) as pd.Series 
    The output is the \Pi returns pd.Series (N x 1) 
    '''
    imp_rets = delta * covmat.dot(weigths).squeeze() # to get a series from a 1-column dataframe
    imp_rets.name = 'Implied Returns'
    return imp_rets

def omega_uncertain_prior(covmat, tau, P):
    '''
    Returns the He-Litterman simplified Omega matrix in case the investor does not explicitly 
    quantify the uncertainty on the views. This matrix is going to be:
       \Omega := diag( P(\tau\Sigma)P^T ) 
    Inputs:
    - covmat: N x N covariance Matrix as pd.DataFrame (\Sigma)
    - tau: a scalar denoting the uncertainty of the CAPM prior
    - P: the Projection K x N matrix as a pd.DataFrame.
    The output is a P x P matrix as a pd.DataFrame, representing the Prior Uncertainties.
    '''
    he_lit_omega = P.dot(tau * covmat).dot(P.T)
    # Make a diag matrix from the diag elements of Omega
    return pd.DataFrame( np.diag(np.diag(he_lit_omega.values)), index=P.index, columns=P.index )

def black_litterman(w_prior, Sigma_prior, P, Q, Omega=None, delta=2.5, tau=0.02):
    '''
    Black-Litterman model.
    Computes the posterior expected returns and covariaces based on the original Black-Litterman model 
    using the Master formulas, where:
    - w_prior is the N x 1 pd.Series of prior weights
    - Sigma_prior is the N x N covariance matrix as a pd.DataFrame
    - P is the projection K x N matrix of weights portfolio views, a pd.DataFrame
    - Q is the K x 1 pd.Series of views
    - Omega is the K x K matrix as a pd.DataFrame (or None) representing the uncertainty of the views. 
      In particualar, if Omega=None, we assume that it is proportional to variance of the prior (see OMEGA_UNCERTAIN_PRIOR).
    - delta is the risk aversion (scalar)
    - tau represents the uncertainty of the CAPM prior (scalar)
    '''
    if Omega is None:
        Omega = omega_uncertain_prior(Sigma_prior, tau, P)
    
    # number of assets
    N = w_prior.shape[0]

    # number of views
    K = Q.shape[0]
    
    # First, reverse-engineer the weights to get \Pi = \delta\Sigma\w^{prior}
    Pi = implied_returns(Sigma_prior,  w_prior, delta)
        
    # Black-Litterman posterior estimate (Master Formulas), using the versions that do not require Omega to be inverted
    invmat   = inv( P.dot(tau * Sigma_prior).dot(P.T) + Omega )
    mu_bl    = Pi + (tau * Sigma_prior).dot(P.T).dot(invmat.dot(Q - P.dot(Pi).values))
    sigma_bl = Sigma_prior + (tau * Sigma_prior) - (tau * Sigma_prior).dot(P.T).dot(invmat).dot(P).dot(tau * Sigma_prior)
    return (mu_bl, sigma_bl)

def posterior_weights(bl_returns, bl_cov, delta=2.5):
    '''
    Computes the posterior weights from the Black-Litterman expected returns and covariances.
    The inputs are:
    - bl_returns: the posterior expected returns as a pd.Series
    - bl_cov: the posterior covariance matrix as a pd.DataFrame
    - delta: the risk aversion coefficient
    The output is the posterior weights as a pd.Series
    '''
    cov_inv = delta * np.linalg.inv(bl_cov.values)
    posterior_weights = np.dot(cov_inv, bl_returns)/np.sum(np.dot(cov_inv, bl_returns))
    posterior_weights = pd.Series(posterior_weights, index=bl_returns.index)
    return posterior_weights

In [34]:
tickers = ['INTC', 'PFE']
Sigma = pd.DataFrame([[46.0, 1.06], [1.06, 5.33]], index=tickers, columns=tickers) * 10E-4

w_prior = pd.Series([0.44, 0.56], index=tickers)

mu_exp = pd.Series([0.02, 0.04], index=tickers)
# w_prior = np.linalg.inv(Sigma).dot(mu_exp)/np.sum(np.linalg.inv(Sigma).dot(mu_exp))


Q = pd.Series([0.02, 0.04], index=tickers)

P = pd.DataFrame( np.array([[1,0],[0,1]]), columns=tickers )

In [35]:
w_prior

INTC    0.44
PFE     0.56
dtype: float64

In [36]:
ret,cov = black_litterman(w_prior, Sigma, P, Q)

In [37]:
ret,cov

(INTC    0.037622
 PFE     0.024111
 dtype: float64,
           INTC       PFE
 INTC  0.046459  0.001065
 PFE   0.001065  0.005383)

In [ ]:
# USE SOME OTHER METHOD OR OPTIMIZATION TO GET THE WEIGHTS
posterior_weights(ret, cov)

INTC    0.140692
PFE     0.859308
dtype: float64

In [14]:
posterior_weights = np.dot(np.linalg.inv(cov), ret)/np.sum(np.dot(np.linalg.inv(cov), ret))
posterior_weights

array([0.14069213, 0.85930787])